# Partie 1

*AUDIN Loric, FAIVRE Elio, PERREY PRATO Joan, RUFFIN Axel*

## 1.Contexte
L’ADEME (Agence de l’Environnement et de la Maîtrise de l’Énergie) souhaiterait ralentir le réchauffement climatique en essayant de diviser par 4 les émissions d'ici 2050 pour la France. Pour cela, elle aimerait optimiser la logistique du transport en limitant au maximum les déplacements et la consommation des véhicules lors des livraisons.

Nous, CesiCDP, devons les aider à atteindre les objectifs et permettre de trouver les trajets optimaux rapidement dans toute la France. Il leur faudrait une solution pour résoudre leur problème d'optimisation ainsi que des études de celles-ci.

## 2. Identification du problème

Ce problème est modélisé comme une variante du Traveling Salesman Problem.

En intégrant des contraintes supplémentaires, notamment des fenêtres temporelles, on obtient une variante appelée Traveling Salesman Problem with Time Windows.

Pour répondre aux objectifs de l'ADEME, notre modèle ne se contente pas de chercher le chemin le plus court car il doit intégrer les réalités logistiques via deux contraintes majeures :

- Coût ou restriction de passage sur certaines arêtes : Contrairement à un modèle simplifié où toutes les routes sont accessibles, notre modèle intègre des pondérations variables sur les arêtes ($c_{ij}$). Cela nous permet de modéliser des travaux programmés ou des restrictions de gabarit. Certaines arêtes peuvent être totalement interdites (coût M), forçant l'algorithme à recalculer un détour optimal.

- Fenêtres temporelles (Time Windows) : Chaque ville $i$ est associée à un intervalle de temps $[e_i, l_i]$. Le véhicule doit impérativement arriver avant l'heure limite $l_i$. S'il arrive avant l'heure d'ouverture $e_i$, il doit patienter, ce qui impacte le temps total de la tournée. Cette contrainte est capitale pour modéliser les horaires de livraison en zone urbaine ou les créneaux de réception des clients.

Pour la représentation du réseau routier national sous forme de graphe, nous considérerons que les noeuds sont des villes et les arêtes sont les routes.


## 3. Définition mathématique du problème

### Formulation précise

On établit le problème suivant :
Soit un graphe non orienté $G(S, A)$ avec : <br>
- $S = \{s_0, s_1, \dots, s_n\}$ est l'ensemble des sommets,<br>
- $A$ l'ensemble des arêtes,<br>
- $c_{ij}$ le coût de la traversée de l'arête $(s_i, s_j)$, il représente la distance en km.
- $d_{ij}$ le temps de trajet nécéssaire pour parcourir l'arête $(s_i, s_j)$.
- $[e_i, l_i]$ la fenêtre temporelle de la ville $s_i$, où $e_i$ est l'heure d'arrivée au plus tôt et $l_i$ l'heure au plus tard. ($e$ pour "earliest" et $l$ pour "latest")

### Fonction objectif

$$ \quad Z = \sum_{(i,j) \in A} c_{ij} \cdot x_{ij}$$

$$
\begin{cases}
\text{Minimiser } Z = \displaystyle\sum_{i \in S} \displaystyle\sum_{j \in S} c_{ij} \cdot x_{ij} \\
\\
\text{Sujet à :} \\
\displaystyle\sum_{j \in S, j \neq i} x_{ij} = 1, \quad \forall i \in S & \text{(Chaque ville est quittée une fois)} \\
\displaystyle\sum_{i \in S, i \neq j} x_{ij} = 1, \quad \forall j \in S & \text{(Chaque ville est visitée une fois)} \\
t_i + d_{ij} - t_j \le M(1 - x_{ij}), \quad \forall (i,j) \in A, j \neq 1 & \text{(Contrainte MTZ)} \\
e_i \le t_i \le l_i, \quad \forall i \in S & \text{(Respect des fenêtres temporelles)} \\
x_{ij} \in \{0, 1\}, \quad \forall (i,j) \in A & \text{(Variable binaire)}
\end{cases}
$$

## 4.Explicitation des contraintes (Avec équations)

Pour transformer le problème du voyageur de commerce classique(TSP) en un problème avec fenêtres temporelles (TSPTW), nous devons introduire une variable de décision supplémentaire :
- $t_i$ : l'instant auquel le véhicule commence à servir la ville $i$.
### A. Contrainte de Fenêtre Temporelle
Pour chaque sommet $i \in S$, le passage doit s'effectuer dans l'intervalle imparti :$$e_i \le t_i \le l_i$$
- $e_i$ : heure d'ouverture au plus tôt (earliest).
- $l_i$ : heure de fermeture au plus tard (latest).
### B. Contrainte de cohérence chronologique
Si le véhicule se déplace de la ville $i$ vers la ville $j$ ($x_{ij} = 1$), alors l'heure d'arrivée en $j$ doit être supérieure ou égale à l'heure de départ de $i$ augmentée du temps de trajet $d_{ij}$ :$$t_i + d_{ij} \le t_j + M(1 - x_{ij})$$(Où $M$ est une constante très grande permettant d'activer/désactiver la contrainte selon la valeur de $x_{ij}$).
- Si $x_{ij} = 1$ : la contrainte devient $t_i + d_{ij} \le t_j$ (le temps de trajet est respecté).
### C. Restriction de passage sur les arêtes
Le réseau routier peut présenter des restrictions (travaux, interdictions). Pour modéliser l'impossibilité d'emprunter une route entre $i$ et $j$, on définit un coût $c_{ij}$ prohibitif ou on retire l'arête de l'ensemble des arcs admissibles $A$ :$$c_{ij} = Mc_{ij} $$Cette restriction force l'algorithme à chercher un chemin alternatif, impactant directement la distance totale et le respect des fenêtres temporelles des villes suivantes.

Notre but est de trouver un cycle hamiltonien qui traverse chaque sommet une seule fois, tout en minimisant la somme des coûts de traversée

## 5. Complexité

Pour démontrer la complexité de notre problème d'optimisation de tournées avec restrictions de passage et des fenetres temporelles, nous allons procéder par étape : définir le problème de décision de base, déterminer le problème de décision du TSP,  comparer avec notre problème TSPTW, et démontrer que le problème est NP-difficile.

### 5.1. Complexité du problème de décision de base (TSP)
Notre étude s'appuie fondamentalement sur le Problème du Voyageur de Commerce (TSP - Traveling Salesperson Problem). Pour évaluer sa complexité théorique, nous devons d'abord considérer sa version de décision (TSP-D) :

- Données : graphe complet pondéré G=(S,A,c) (où S est l'ensemble des sommets, A l'ensemble des arêtes et c la fonction de coût associée) et un réel K >= 0.
- Question : Existe-t-il un cycle hamiltonien dont le coût total est inférieur ou égal à K ?



#### Preuve de la NP-complétude (Réduction de Karp)
Il a été formellement démontré par Richard M. Karp en 1972 que le problème du cycle hamiltonien est NP-complet. Le problème du TSP de décision (TSP-D) est la réduction polynomiale depuis le problème du cycle hamiltonien, il est lui-même classé NP-complet. Par conséquent, le problème d'optimisation consistant à trouver la tournée minimale (le TSP classique) est formellement NP-difficile. [1]



#### Appartenance à la classe NP (Algorithme de certificat)
Cependant, pour prouver que le problème de décision (TSP-D) appartient à la classe NP, nous devons démontrer qu'une solution (un certificat) peut être vérifiée en temps polynomial. Pour un cycle candidat $C$, les étapes de l'algorithme de certificat sont les suivantes :

- S'assurer que la séquence $C$ forme bien un chemin continu dans le graphe et qu'elle contient exactement $n$ arêtes (pour relier $n$ sommets et revenir au point de départ). Cette vérification se fait en $O(n)$.
- S'assurer que chaque sommet du graphe n’apparaît qu'une seule fois dans le parcours (à l'exception du sommet de départ/arrivée). Il suffit d’implémenter un dictionnaire ou un tableau associant à chaque sommet son état de visite. Cette étape se fait en $O(n)$.
- S'assurer que les extrémités du parcours (le sommet de départ $u$ et le sommet d'arrivée $v$) sont identiques ($u = v$). Cette vérification se fait en $O(1)$.
- Parcourir les arêtes du cycle $C$, additionner leurs coûts respectifs, et vérifier que le coût total est bien inférieur ou égal à la limite fixée $K$. Cette somme nécessite de lire les $n$ arêtes, ce qui se fait en $O(n)$.

Chaque étape étant de complexité polynomiale, l’algorithme de vérification est lui-même polynomial. Le problème est donc bien dans NP.

### 5.2 Adaptation à notre cas : TSP Time-Windows (TSPTW)
Dans le cadre de notre projet pour l'ADEME, le problème n'est pas un TSP statique classique. Nous rappelons que nous avons ajouté deux contraintes majeures qui modifient le graphe G :

- Coût ou restriction de passage sur certaines arêtes
- Fenêtres temporelles (Time Windows)

Notre problème est donc une variante du TSP avec des fenêtres temporelles (TSPTW) puisque dans notre cas, la circulation sont en fonction du coût des arêtes et de la disponibilité des sommets.

### 5.3 Preuve de la NP-difficulté de notre problème

Puisqu'il est établi que le TSP est un problème NP-difficile , notre problème TSPTW inculant des contraintes supplémentaires comme des fenetres de temps et des restrictions de passage est par conséquent au moins aussi difficile à résoudre, ce qui confirme sa NP-difficulté.

## 6. Choix des algorithmes

Nous avons fait le choix de la métaheuristique des fourmis artificiels
(À COMPLÉTER)

## 7. Documentation de l'étude

Note : Référence fondatrice démontrant la NP-complétude du problème du voyageur de commerce via la réduction du problème du cycle hamiltonien.

- [1] KARP, Richard M. Reducibility among combinatorial problems In Complexity of computer computations. Boston, MA : Springer, 1972. p. 85-103. [Disponible ici](https://cgi.di.uoa.gr/~sgk/teaching/grad/handouts/karp.pdf)

# Partie 2

## 1. Instanciation du graphe

### 1.1. Structure du graphe
Le graphe sera représenté sous la forme de matrices d'adjacences et de liste de tuples :

$
matrice = \begin{pmatrix}
    s_{1, 1} & s_{1, 2} & ... & s_{1, 1} \\
    s_{2, 1} & s_{2, 2} & ... & s_{1, 1} \\
    ... &... & ... & ... \\
    s_{n, 1} & s_{n, 2} & ... & s_{n, n}
\end{pmatrix}
$

avec $s_{i, j}$ la distance entre la ville i et j.

$liste\_villes = [(e_1, l_1), (e_2, l_2), ..., (e_n, l_n)]$

avec :
- e<sub>i</sub> : heure d'arrivée au plus tôt
- l<sub>i</sub> : heure d'arrivée au plus tard

On aura donc la structure suivante :

In [ ]:
import numpy as np

NB_VILLES = 5
POURCENTAGE_ROUTES_BLOQUEES = 0

matrice = np.zeros((NB_VILLES, NB_VILLES), tuple)
liste_villes = []

print(matrice)

[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]


### 1.2. Génération aléatoire du graphe
Afin de générer un graphe respectant l'inégalité triangulaire et contenant au moins un cycle hamiltonien, nous choisissons d'associer à chaque ville des coordonnées, pour ensuite calculer la distance entre les points pour les arêtes. Pour les routes que l'on ne peut pas emprunter, on mettra la valeur M (c'est-à-dire le plus grand entier possible). On aura le code suivant :

In [ ]:
import random

random.seed(a=3)

coordonnees_villes = [(random.randint(1, 1000), random.randint(1, 1000)) for _ in range(NB_VILLES)]
print("Coordonnées des villes :", coordonnees_villes)

Coordonnées des villes : [(244, 607), (558, 134), (379, 938), (619, 486), (641, 595)]


Toutes les villes seront reliées entre elles (le graphe est donc complet). Les arêtes auront une distance calculée en fonction de leurs positions avec la formule suivante :

$
distance = \sqrt{(|x2 - x1|)^2 + (|y2 - y1|)^2}
$

Avec ceci, on aura l'inégalité triangulaire qui sera respecté dans notre graphe généré (pour simplifier la lisibilité des résultats, nous ne garderons que les parties entières des résultats).

In [ ]:
from math import sqrt
from sys import maxsize

for i in range(NB_VILLES):
    matrice[i][i] = (0,0)
    for j in range(i + 1, NB_VILLES):
        if random.randint(1, 100) <= POURCENTAGE_ROUTES_BLOQUEES:
            distance = maxsize
            temps = maxsize
        else:
            distance = round(sqrt(abs(coordonnees_villes[j][0] - coordonnees_villes[i][0]) ** 2 + abs(coordonnees_villes[j][1] - coordonnees_villes[i][1]) ** 2))
            temps = round(distance / (random.randint(10, 100)))
        
        arete = (distance, temps)
        matrice[i][j] = arete
        matrice[j][i] = arete

print(matrice)

[[(0, 0) (568, 13) (357, 4) (394, 7) (397, 7)]
 [(568, 13) (0, 0) (824, 14) (357, 5) (468, 8)]
 [(357, 4) (824, 14) (0, 0) (512, 6) (432, 5)]
 [(394, 7) (357, 5) (512, 6) (0, 0) (111, 1)]
 [(397, 7) (468, 8) (432, 5) (111, 1) (0, 0)]]


Et on crée nos villes (si les heures sont les mêmes pour l'ouverture et la fermeture, la ville est accessible 24h/24) :

In [ ]:
for i in range(NB_VILLES):
    ville = (random.randint(0, 23), random.randint(0, 23))
    liste_villes.append(ville)
print(liste_villes)

[(9, 18), (19, 10), (5, 11), (5, 10), (11, 19), (8, 9), (12, 3), (0, 18), (21, 23), (4, 9)]


## 2. Instanciation des algorithmes

## 3. Plan d'expérience

## 4. Étude expérimentale
### Étude statistique

### Pistes d'amélioration

# Conclusion